In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV
from sklearn import metrics

In [2]:
np.random.seed(42)

r2_score_list = []
rmse_score_list = []
for i in range(10):
    data = pd.read_csv('data/data_DFT_mordredpca.csv')
    data['Alc_ID'] = data.index // 14
    shuffled_groups = data['Alc_ID'].unique()
    np.random.shuffle(shuffled_groups)
    train_groups = shuffled_groups[:10]
    test_groups = shuffled_groups[10:]
    train_data = data[data['Alc_ID'].isin(train_groups)].reset_index(drop=True)
    test_data = data[data['Alc_ID'].isin(test_groups)].reset_index(drop=True)

    y_train = pd.DataFrame(train_data['Yield'],columns=['Yield'])
    X_train = train_data.drop(columns=['PC_ID', 'Alc_ID', 'Yield', 'PC_SMILES', 'Alc_SMILES'])
    y_test = pd.DataFrame(test_data['Yield'],columns=['Yield'])
    X_test = test_data.drop(columns=['PC_ID', 'Alc_ID', 'Yield', 'PC_SMILES', 'Alc_SMILES'])
    
    a_X_train = (X_train - X_train.mean()) / X_train.std()
    a_X_test = (X_test - X_train.mean()) / X_train.std()
    a_X_train = a_X_train.dropna(how='any', axis=1)
    a_X_test = a_X_test[a_X_train.columns]
    reg = RidgeCV(alphas=np.linspace(0.1, 30, num=150), cv=5)
    reg.fit(a_X_train, y_train['Yield'])
    y_pred1 = reg.predict(a_X_train)
    y_pred2 = reg.predict(a_X_test)
    r2 = metrics.r2_score(y_test, y_pred2)
    rmse = metrics.root_mean_squared_error(y_test, y_pred2)
    print(f'Run{i} R2 (test):', r2, ', RMSE (test):', rmse)
    r2_score_list.append(r2)
    rmse_score_list.append(rmse)
print('==========(Result)==========')
print('Mean R2:', np.mean(r2_score_list))
print('SD R2:', np.std(r2_score_list))
print('Mean RMSE:', np.mean(rmse_score_list))
print('SD RMSE:', np.std(rmse_score_list))

Run0 R2 (test): -1.986590298595504 , RMSE (test): 19.08809695122195
Run1 R2 (test): 0.30859058794874095 , RMSE (test): 23.03115855824047
Run2 R2 (test): 0.2733530942115996 , RMSE (test): 26.12857397665477
Run3 R2 (test): -0.04111615112191669 , RMSE (test): 26.941483606917828
Run4 R2 (test): -0.38642031899804774 , RMSE (test): 21.789396077632674
Run5 R2 (test): 0.40731826597805576 , RMSE (test): 20.68370536777944
Run6 R2 (test): -0.2324974378645448 , RMSE (test): 25.2360860216531
Run7 R2 (test): 0.20828773195311745 , RMSE (test): 21.321927974099737
Run8 R2 (test): -0.3110360603261444 , RMSE (test): 18.968008199773585
Run9 R2 (test): -0.6772978222719954 , RMSE (test): 14.739917157307191
==========(Result)==========
Mean R2: -0.24374084090866394
SD R2: 0.6702409330741523
Mean RMSE: 21.792835389128076
SD RMSE: 3.538184224119285
